# Delta Lake Optimize and Vacuum

This notebook demonstrates Delta Lake's file management capabilities. We'll:

1. Run OPTIMIZE commands on the Delta table to compact small files
2. Implement Z-ORDER BY for data co-location
3. Collect and display statistics before and after optimization
4. Run VACUUM with appropriate retention period
5. Demonstrate the impact on query performance with benchmarks
6. Monitor file counts and sizes before and after operations
7. Implement best practices for scheduling these maintenance operations
8. Include safety checks before running destructive operations

## 1. Initialize Spark Session with Delta Lake

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, lit, current_timestamp
from delta.tables import DeltaTable

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
import utils

# Create Spark session with Delta Lake support
spark = SparkSession.builder \
    .appName("Delta Lake Optimize and Vacuum") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

FileNotFoundError: [Errno 2] No such file or directory: '/opt/spark/./bin/spark-submit'

## 2. Load Delta Table and Get Initial Metrics

In [ ]:
# Define Delta table path
delta_table_path = "/opt/spark/data/processed/global_superstore_delta"

# Load the Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# Get initial table metrics
initial_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
initial_version = initial_metrics["current_version"]

print(f"Initial table version: {initial_version}")
print(f"Initial record count: {initial_metrics['record_count']}")
print(f"Initial file count: {initial_metrics['num_files']}")
print(f"Initial table size: {initial_metrics['size_in_mb']:.2f} MB")

## 3. Generate Small Files Problem

To demonstrate the benefits of OPTIMIZE, let's first create a small files problem by writing many small batches to the Delta table.

In [ ]:
# Function to generate and write a small batch of data
def write_small_batch(batch_id):
    # Generate a small batch of data
    batch_data = utils.generate_test_data(num_records=50, scenario='normal')
    
    # Convert to Spark DataFrame
    batch_df = spark.createDataFrame(batch_data)
    
    # Add batch identifier
    batch_df = batch_df.withColumn("Batch_ID", lit(f"small_batch_{batch_id}"))
    
    # Write to Delta table
    batch_df.write \
        .format("delta") \
        .mode("append") \
        .save(delta_table_path)
    
    return batch_df.count()

# Write multiple small batches
print("Writing small batches to create a small files problem...")
total_records = 0
num_batches = 20

for i in range(num_batches):
    records = write_small_batch(i)
    total_records += records
    print(f"Wrote batch {i+1}/{num_batches} with {records} records")

print(f"Wrote {total_records} records in {num_batches} small batches")

# Get metrics after creating small files
small_files_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
print(f"\nAfter creating small files:")
print(f"Table version: {small_files_metrics['current_version']}")
print(f"Record count: {small_files_metrics['record_count']}")
print(f"File count: {small_files_metrics['num_files']}")
print(f"Table size: {small_files_metrics['size_in_mb']:.2f} MB")
print(f"Files added: {small_files_metrics['num_files'] - initial_metrics['num_files']}")

## 4. Benchmark Queries Before Optimization

In [ ]:
# Define benchmark queries
def query1(spark):
    # Simple filter query
    return spark.read.format("delta").load(delta_table_path) \
        .filter(col("Ship Mode") == "First Class") \
        .count()

def query2(spark):
    # Aggregation query
    return spark.read.format("delta").load(delta_table_path) \
        .groupBy("Category", "Ship Mode") \
        .agg({"Sales": "sum", "Profit": "sum"}) \
        .count()

def query3(spark):
    # Complex query with multiple filters
    return spark.read.format("delta").load(delta_table_path) \
        .filter((col("Ship Mode") == "First Class") & 
                (col("Category") == "Furniture") & 
                (col("Sales") > 100)) \
        .orderBy(col("Sales").desc()) \
        .count()

# Run benchmarks before optimization
print("Benchmarking queries before optimization...")
before_query1 = utils.benchmark_query(spark, query1, num_runs=5)
before_query2 = utils.benchmark_query(spark, query2, num_runs=5)
before_query3 = utils.benchmark_query(spark, query3, num_runs=5)

# Store results for comparison
before_results = {
    "query1": before_query1["avg_time"],
    "query2": before_query2["avg_time"],
    "query3": before_query3["avg_time"]
}

print("\nBenchmark results before optimization:")
for query, time in before_results.items():
    print(f"{query}: {time:.4f} seconds")

## 5. Run OPTIMIZE Command

In [ ]:
# Monitor file metrics before optimization
before_optimize_metrics = utils.monitor_file_metrics(spark, delta_table_path, "before")

# Run OPTIMIZE command
print("Running OPTIMIZE command...")
optimize_start_time = time.time()

spark.sql(f"OPTIMIZE delta.`{delta_table_path}`").show()

optimize_end_time = time.time()
optimize_duration = optimize_end_time - optimize_start_time
print(f"OPTIMIZE completed in {optimize_duration:.2f} seconds")

# Monitor file metrics after optimization
after_optimize_metrics = utils.monitor_file_metrics(spark, delta_table_path, "after")

# Calculate improvement
file_reduction = before_optimize_metrics["num_files"] - after_optimize_metrics["num_files"]
file_reduction_pct = (file_reduction / before_optimize_metrics["num_files"]) * 100

print(f"\nOptimization results:")
print(f"Files before: {before_optimize_metrics['num_files']}")
print(f"Files after: {after_optimize_metrics['num_files']}")
print(f"Files reduced: {file_reduction} ({file_reduction_pct:.2f}%)")
print(f"Size before: {before_optimize_metrics['size_in_mb']:.2f} MB")
print(f"Size after: {after_optimize_metrics['size_in_mb']:.2f} MB")

## 6. Run Z-ORDER BY Command

In [ ]:
# Run Z-ORDER BY command
print("Running Z-ORDER BY command...")
zorder_start_time = time.time()

spark.sql(f"OPTIMIZE delta.`{delta_table_path}` ZORDER BY (`Order Date`, `Customer ID`, `Product ID`)").show()

zorder_end_time = time.time()
zorder_duration = zorder_end_time - zorder_start_time
print(f"Z-ORDER completed in {zorder_duration:.2f} seconds")

# Get metrics after Z-ORDER
after_zorder_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
print(f"\nAfter Z-ORDER:")
print(f"Table version: {after_zorder_metrics['current_version']}")
print(f"File count: {after_zorder_metrics['num_files']}")
print(f"Table size: {after_zorder_metrics['size_in_mb']:.2f} MB")

## 7. Benchmark Queries After Optimization

In [ ]:
# Run benchmarks after optimization
print("Benchmarking queries after optimization...")
after_query1 = utils.benchmark_query(spark, query1, num_runs=5)
after_query2 = utils.benchmark_query(spark, query2, num_runs=5)
after_query3 = utils.benchmark_query(spark, query3, num_runs=5)

# Store results for comparison
after_results = {
    "query1": after_query1["avg_time"],
    "query2": after_query2["avg_time"],
    "query3": after_query3["avg_time"]
}

print("\nBenchmark results after optimization:")
for query, time in after_results.items():
    print(f"{query}: {time:.4f} seconds")

# Calculate improvement
print("\nPerformance improvement:")
for query in before_results.keys():
    improvement = before_results[query] - after_results[query]
    improvement_pct = (improvement / before_results[query]) * 100
    print(f"{query}: {improvement:.4f} seconds faster ({improvement_pct:.2f}%)")

# Visualize the results
plt.figure(figsize=(10, 6))
queries = list(before_results.keys())
before_times = [before_results[q] for q in queries]
after_times = [after_results[q] for q in queries]

x = np.arange(len(queries))
width = 0.35

plt.bar(x - width/2, before_times, width, label='Before Optimization')
plt.bar(x + width/2, after_times, width, label='After Optimization')

plt.xlabel('Query')
plt.ylabel('Execution Time (seconds)')
plt.title('Query Performance Before and After Optimization')
plt.xticks(x, queries)
plt.legend()

plt.tight_layout()
plt.show()

## 8. Examine Delta Table History

In [ ]:
# Examine Delta table history
history = delta_table.history(10).toPandas()
print("Delta table history (last 10 versions):")
history

## 9. Run VACUUM Command with Safety Checks

In [ ]:
# Check current retention period
retention_period = spark.sql(f"SHOW TBLPROPERTIES delta.`{delta_table_path}` ('delta.deletedFileRetentionDuration')").collect()
print(f"Current retention period: {retention_period[0][1]}")

# Safety check: Ensure we have a reasonable retention period
# If not set, the default is 7 days
if retention_period[0][1] == "null":
    print("Setting retention period to 30 days for safety")
    spark.sql(f"ALTER TABLE delta.`{delta_table_path}` SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days')")

# Temporarily disable retention check for demonstration
# WARNING: In production, never set this below 7 days, especially with streaming workloads
print("\nWARNING: Temporarily disabling retention check for demonstration purposes.")
print("In production, never set retention period below 7 days, especially with streaming workloads.")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# First, run VACUUM DRY RUN to see what would be deleted
print("\nRunning VACUUM DRY RUN to see what would be deleted...")
vacuum_dry_run = spark.sql(f"VACUUM delta.`{delta_table_path}` RETAIN 0 HOURS DRY RUN").collect()
num_files_to_delete = len(vacuum_dry_run)
print(f"Number of files that would be deleted: {num_files_to_delete}")

# Show some of the files that would be deleted
if num_files_to_delete > 0:
    print("Sample of files that would be deleted:")
    for i, row in enumerate(vacuum_dry_run[:5]):
        print(f"  {i+1}. {row[0]}")
    if num_files_to_delete > 5:
        print(f"  ... and {num_files_to_delete - 5} more files")

# Run actual VACUUM
print("\nRunning VACUUM command...")
vacuum_start_time = time.time()

spark.sql(f"VACUUM delta.`{delta_table_path}` RETAIN 0 HOURS").show()

vacuum_end_time = time.time()
vacuum_duration = vacuum_end_time - vacuum_start_time
print(f"VACUUM completed in {vacuum_duration:.2f} seconds")

# Reset retention check for safety
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

# Reset retention period to a safe value
print("\nResetting retention period to 30 days for safety")
spark.sql(f"ALTER TABLE delta.`{delta_table_path}` SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days')")

## 10. Best Practices for Scheduling Maintenance Operations

In [ ]:
# Create a function to demonstrate a maintenance job
def run_maintenance_job(table_path, optimize=True, vacuum=True, retention_hours=168):
    """
    Run a maintenance job on a Delta table.
    
    Parameters:
    - table_path: Path to the Delta table
    - optimize: Whether to run OPTIMIZE
    - vacuum: Whether to run VACUUM
    - retention_hours: Retention period for VACUUM in hours (default: 168 hours = 7 days)
    """
    print(f"Starting maintenance job for {table_path}")
    start_time = time.time()
    
    # Get table metrics before maintenance
    before_metrics = utils.monitor_file_metrics(spark, table_path, "before")
    
    # Run OPTIMIZE if requested
    if optimize:
        print("Running OPTIMIZE...")
        optimize_start = time.time()
        
        # Check if table is large enough to benefit from OPTIMIZE
        if before_metrics["num_files"] > 10:
            # For large tables, consider optimizing only specific partitions
            # Here we're optimizing the entire table for simplicity
            spark.sql(f"OPTIMIZE delta.`{table_path}` ZORDER BY (`Order Date`, `Customer ID`, `Product ID`)").show()
        else:
            print("Table has few files, skipping OPTIMIZE")
            
        optimize_end = time.time()
        print(f"OPTIMIZE completed in {optimize_end - optimize_start:.2f} seconds")
    
    # Run VACUUM if requested
    if vacuum:
        print("\nRunning VACUUM...")
        vacuum_start = time.time()
        
        # Safety check: Ensure retention period is at least 7 days for production
        if retention_hours < 168:  # 168 hours = 7 days
            print("WARNING: Retention period is less than 7 days. This is not recommended for production.")
            print("Setting retention period to 7 days for safety.")
            retention_hours = 168
        
        # Run VACUUM DRY RUN first
        vacuum_dry_run = spark.sql(f"VACUUM delta.`{table_path}` RETAIN {retention_hours} HOURS DRY RUN").collect()
        num_files_to_delete = len(vacuum_dry_run)
        print(f"Number of files that would be deleted: {num_files_to_delete}")
        
        # Run actual VACUUM if there are files to delete
        if num_files_to_delete > 0:
            spark.sql(f"VACUUM delta.`{table_path}` RETAIN {retention_hours} HOURS").show()
        else:
            print("No files to delete, skipping VACUUM")
            
        vacuum_end = time.time()
        print(f"VACUUM completed in {vacuum_end - vacuum_start:.2f} seconds")
    
    # Get table metrics after maintenance
    after_metrics = utils.monitor_file_metrics(spark, table_path, "after")
    
    # Calculate improvements
    file_reduction = before_metrics["num_files"] - after_metrics["num_files"]
    size_change = after_metrics["size_in_mb"] - before_metrics["size_in_mb"]
    
    end_time = time.time()
    total_duration = end_time - start_time
    
    print(f"\nMaintenance job completed in {total_duration:.2f} seconds")
    print(f"Files before: {before_metrics['num_files']}, Files after: {after_metrics['num_files']}")
    print(f"Files reduced: {file_reduction}")
    print(f"Size before: {before_metrics['size_in_mb']:.2f} MB, Size after: {after_metrics['size_in_mb']:.2f} MB")
    print(f"Size change: {size_change:.2f} MB")
    
    return {
        "table_path": table_path,
        "duration": total_duration,
        "file_reduction": file_reduction,
        "size_change": size_change,
        "before_metrics": before_metrics,
        "after_metrics": after_metrics
    }

# Demonstrate the maintenance job
print("Demonstrating a scheduled maintenance job...\n")
maintenance_result = run_maintenance_job(delta_table_path, optimize=True, vacuum=True, retention_hours=168)

## 11. Maintenance Job Scheduling Recommendations

In [ ]:
print("Recommendations for scheduling maintenance operations:")
print("\n1. OPTIMIZE Scheduling:")
print("   - For tables with frequent small writes: Run daily during off-peak hours")
print("   - For tables with moderate write volume: Run weekly")
print("   - For tables with low write volume: Run monthly")
print("   - Consider optimizing only specific partitions for very large tables")
print("   - Monitor file counts and sizes to determine optimal frequency")

print("\n2. VACUUM Scheduling:")
print("   - Always run after OPTIMIZE operations")
print("   - Never set retention period below 7 days, especially with streaming workloads")
print("   - For most workloads: Run weekly with 7-30 day retention")
print("   - For storage-constrained environments: Run more frequently but maintain 7+ day retention")
print("   - Always run DRY RUN first to see what would be deleted")

print("\n3. Z-ORDER Considerations:")
print("   - Z-ORDER is resource-intensive, use selectively")
print("   - Choose columns that are frequently used in filters and joins")
print("   - For most workloads: Run monthly or quarterly")
print("   - Monitor query performance to determine if Z-ORDER is beneficial")

print("\n4. Implementation Options:")
print("   - Use orchestration tools like Airflow, Azure Data Factory, or AWS Glue")
print("   - Create a maintenance job that runs on a schedule")
print("   - Include monitoring and alerting for maintenance job failures")
print("   - Log maintenance job metrics for trend analysis")

## 12. Summary

In this notebook, we've demonstrated Delta Lake's file management capabilities:

1. Using OPTIMIZE to compact small files and improve query performance
2. Implementing Z-ORDER BY for data co-location
3. Running VACUUM to remove old files and reclaim storage
4. Measuring the impact on query performance
5. Implementing best practices for maintenance operations

These operations are essential for maintaining optimal performance and storage efficiency in Delta Lake tables, especially for tables with frequent updates or streaming workloads.